In [22]:
import numpy as np
import pandas as pd
import os
import sys
import zipfile
import subprocess

from matplotlib import pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from scipy import stats
from tqdm.notebook import tqdm
from copy import deepcopy

import json

In [23]:
DATASET = 'ml-1m' 
RAW_PATH = os.path.join('./', DATASET)

RANDOM_SEED = 0
NEG_ITEMS = 99

# Load data

1. Load interaction data and item metadata
2. Filter out items with less than 5 interactions
3. Calculate basic statistics

In [24]:
import requests

# download data if not exists

url = f'http://files.grouplens.org/datasets/movielens/{DATASET}.zip'
zip_path = os.path.join(RAW_PATH, DATASET + '.zip')

os.makedirs(RAW_PATH, exist_ok=True)
if not os.path.exists(zip_path):
    # 1. 下载
    print('Downloading', url)
    r = requests.get(url, stream=True, timeout=30)
    r.raise_for_status()
    with open(zip_path, 'wb') as f:
        for chunk in r.iter_content(chunk_size=1<<20):
            if chunk:
                f.write(chunk)

    # 2. 解压
    print('Extracting...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(RAW_PATH)
else:
    print('Data already exists.')
print('Done.')

Data already exists.
Done.


In [25]:
# read interaction data
interactions = []
user_freq, item_freq = dict(), dict()
file = os.path.join(RAW_PATH,DATASET,"ratings.dat")
with open(file) as F:
    header = 0
    for line in tqdm(F):
        if header == 1:
            header = 0
            continue
        line = line.strip().split("::")
        uid, iid, rating, time = line[0], line[1], float(line[2]), float(line[3])
        if rating >= 4:
            label = 1
        else:
            label = 0
        interactions.append([uid,time,iid,label])
        if int(label)==1:
            user_freq[uid] = user_freq.get(uid,0)+1
            item_freq[iid] = item_freq.get(iid,0)+1

0it [00:00, ?it/s]

In [26]:
# 5-core filtering
select_uid, select_iid = [],[]
while len(select_uid)<len(user_freq) or len(select_iid)<len(item_freq):
    select_uid, select_iid = [],[]
    for u in user_freq:
        if user_freq[u]>=5:
            select_uid.append(u)
    for i in item_freq:
        if item_freq[i]>=5:
            select_iid.append(i)
    print("User: %d/%d, Item: %d/%d"%(len(select_uid),len(user_freq),len(select_iid),len(item_freq)))

    select_uid = set(select_uid)
    select_iid = set(select_iid)
    user_freq, item_freq = dict(), dict()
    interactions_5core = []
    for line in tqdm(interactions):
        uid, iid, label = line[0], line[2], line[-1]
        if uid in select_uid and iid in select_iid:
            interactions_5core.append(line)
            if int(label)==1:
                user_freq[uid] = user_freq.get(uid,0)+1
                item_freq[iid] = item_freq.get(iid,0)+1
    interactions = interactions_5core

User: 6034/6038, Item: 3125/3533


  0%|          | 0/1000209 [00:00<?, ?it/s]

In [27]:
print("Selected Interactions: %d, Users: %d, Items: %d"%(len(interactions),len(select_uid),len(select_iid)))

Selected Interactions: 994338, Users: 6034, Items: 3125


In [28]:
# Get timestamp
ts = []
for i in tqdm(range(len(interactions))):
    ts.append(datetime.fromtimestamp(interactions[i][1]))

  0%|          | 0/994338 [00:00<?, ?it/s]

In [29]:
# Construct and Save 5 core results with situation context
interaction_df = pd.DataFrame(interactions,columns = ["user_id","time","news_id","label"])
interaction_df['timestamp'] = ts
interaction_df['hour'] = interaction_df['timestamp'].apply(lambda x: x.hour)
interaction_df['weekday'] = interaction_df['timestamp'].apply(lambda x: x.weekday())
interaction_df['date'] = interaction_df['timestamp'].apply(lambda x: x.date())
# ?这啥啊这是？
def get_time_range(hour): # according to the Britannica dictionary
    # https://www.britannica.com/dictionary/eb/qa/parts-of-the-day-early-morning-late-morning-etc
    if hour>=5 and hour<=8:
        return 0
    if hour>8 and hour<11:
        return 1
    if hour>=11 and hour<=12:
        return 2
    if hour>12 and hour<=15:
        return 3
    if hour>15 and hour<=17:
        return 4
    if hour>=18 and hour<=19:
        return 5
    if hour>19 and hour<=21:
        return 6
    if hour>21:
        return 7
    return 8 # 0-4 am

interaction_df['period'] = interaction_df.hour.apply(lambda x: get_time_range(x))
min_date = interaction_df.date.min()
interaction_df['day'] = (interaction_df.date - min_date).apply(lambda x: x.days)


interaction_df["user_id"] = interaction_df["user_id"].astype(int)
interaction_df["item_id"] = interaction_df["news_id"].astype(int)
interaction_df.to_csv("interaction_5core.csv",index=False)


---
# 为DiffRec项目准备数据集
仅需要保留 user_id, item_id, time 三列即可。

In [30]:

GDR_PATH='./ML_1MGDR/'
os.makedirs(GDR_PATH,exist_ok=True)

if os.path.exists('./interaction_5core.csv'):
    interaction_df = pd.read_csv('./interaction_5core.csv')
    print('✅ 读取 interaction 数据集成功')

✅ 读取 interaction 数据集成功


In [31]:
# ==========================================
# 修正后的数据划分逻辑：Leave-Last-Out
# ==========================================

GDR_PATH='./ML_1MGDR/'
os.makedirs(GDR_PATH, exist_ok=True)

# 1. 排序
interaction_gdr = interaction_df[['user_id', 'item_id', 'time']].copy()
interaction_gdr = interaction_gdr.sort_values(['user_id', 'time'])

# 2. 划分 Train/Dev/Test
# 策略：
# Test: 每个用户的最后一次交互
# Dev:  每个用户的倒数第二次交互
# Train: 剩下的所有交互

train_data = []
dev_data = []
test_data = []

# 按用户分组处理
for user_id, group in tqdm(interaction_gdr.groupby('user_id')):
    # 如果交互记录太少（比如少于3条），全放进训练集，或者丢弃
    if len(group) < 3:
        train_data.append(group)
        continue
    
    # 转换为列表
    items = group['item_id'].tolist()
    times = group['time'].tolist()
    
    # Test: 最后一个
    test_data.append([user_id, items[-1], times[-1]])
    
    # Dev: 倒数第二个
    dev_data.append([user_id, items[-2], times[-2]])
    
    # Train: 剩下的
    train_df_user = pd.DataFrame({
        'user_id': [user_id] * (len(items) - 2),
        'item_id': items[:-2],
        'time': times[:-2]
    })
    train_data.append(train_df_user)

# 3. 合并与保存
train_df = pd.concat(train_data, ignore_index=True)
dev_df = pd.DataFrame(dev_data, columns=['user_id', 'item_id', 'time'])
test_df = pd.DataFrame(test_data, columns=['user_id', 'item_id', 'time'])

print(f"Train size: {len(train_df)}")
print(f"Dev size:   {len(dev_df)}")
print(f"Test size:  {len(test_df)}")

# 保存
train_df.to_csv(os.path.join(GDR_PATH, 'train.csv'), index=False, sep='\t')
dev_df.to_csv(os.path.join(GDR_PATH, 'dev.csv'), index=False, sep='\t')
test_df.to_csv(os.path.join(GDR_PATH, 'test.csv'), index=False, sep='\t')

print("✅ 数据处理完成！Train/Dev/Test 包含相同的用户集合。")

  0%|          | 0/6034 [00:00<?, ?it/s]

Train size: 982270
Dev size:   6034
Test size:  6034
✅ 数据处理完成！Train/Dev/Test 包含相同的用户集合。


In [32]:
# # keep only user_id, item_id, time for DiffRec (GDR) and save
# interaction_gdr = interaction_df[['user_id', 'item_id', 'time']].copy()

# # # method 1: sort by user_id and time

# # interaction_gdr = interaction_gdr.sort_values(['user_id', 'time'])
# # seq_interaction_gdr = interaction_gdr.groupby('user_id')['item_id'].apply(list).reset_index()
# # seq_interaction_gdr['last_day'] = interaction_gdr.groupby('user_id')['day'].last().values   # 用于划分

# # # 80% 用户用于训练，10% 验证，10% 测试（用户级划分）
# # split1 = seq_interaction_gdr['last_day'].quantile(0.8)
# # split2 = seq_interaction_gdr['last_day'].quantile(0.9)

# # train_u = seq_interaction_gdr.loc[seq_interaction_gdr['last_day'] <= split1, 'user_id'].values
# # val_u   = seq_interaction_gdr.loc[(seq_interaction_gdr['last_day'] > split1) & 
# #                      (seq_interaction_gdr['last_day'] <= split2), 'user_id'].values
# # test_u  = seq_interaction_gdr.loc[seq_interaction_gdr['last_day'] > split2, 'user_id'].values

# # method 2: random

# # 获取所有唯一的 user_id
# unique_users = interaction_gdr['user_id'].unique()

# # 设置随机种子以保证可重复性
# np.random.seed(42)

# # 打乱 user_id 顺序
# shuffled_users = np.random.permutation(unique_users)

# # 计算划分点
# n_users = len(shuffled_users)
# train_end = int(0.8 * n_users)
# val_end = int(0.9 * n_users)

# # 划分 user_id
# train_users = shuffled_users[:train_end]
# val_users = shuffled_users[train_end:val_end]
# test_users = shuffled_users[val_end:]

# # 创建一个新的列来标记数据集
# def assign_dataset(user_id):
#     if user_id in train_users:
#         return 'train'
#     elif user_id in val_users:
#         return 'dev'
#     else:
#         return 'test'

# interaction_gdr['dataset'] = interaction_gdr['user_id'].apply(assign_dataset)

# # 如果你想拆分成三个独立的 DataFrame
# train_df = interaction_gdr[interaction_gdr['dataset'] == 'train'].drop(columns='dataset')
# dev_df = interaction_gdr[interaction_gdr['dataset'] == 'dev'].drop(columns='dataset')
# test_df = interaction_gdr[interaction_gdr['dataset'] == 'test'].drop(columns='dataset')

# train_df.to_csv(os.path.join(GDR_PATH, 'train.csv'), index=False ,sep='\t')
# dev_df.to_csv(os.path.join(GDR_PATH, 'dev.csv'), index=False ,sep='\t')
# test_df.to_csv(os.path.join(GDR_PATH, 'test.csv'), index=False ,sep='\t')
